## Train Controller

Goal: Train controller on the car racing task

In [1]:
import hashlib
import json
from datetime import datetime
from functools import partial
from pathlib import Path

import gymnasium as gym
import numpy as np
import torch

from world_models.models.vae import VAE
from world_models.models.mdn_rnn import MDNRNN
from world_models.models.controller import Controller
from world_models.envs.carracing import encode_observation
from world_models.evaluation.rollout import rollout

cwd = Path.cwd().resolve()
repo_root = next(
    p for p in (cwd, *cwd.parents)
    if (p / "src" / "world_models").is_dir()
    and (p / "pyproject.toml").is_file()
)

# Start with CPU for single-step inference.
device = torch.device("cpu")

In [2]:
memory_path = (
    repo_root
    / "runs"
    / "memory_pilot_20260910_191458_564311"
    / "best.pt"
)

memory_checkpoint = torch.load(
    memory_path,
    map_location="cpu",
    weights_only=True,
)

latent_metadata = memory_checkpoint["latent_manifest"]
vae_path = Path(latent_metadata["vae_checkpoint"])

actual_hash = hashlib.sha256(vae_path.read_bytes()).hexdigest()
assert actual_hash == latent_metadata["vae_checkpoint_sha256"], (
    "The VAE checkpoint differs from the one used to encode the dataset."
)

vae_checkpoint = torch.load(
    vae_path,
    map_location="cpu",
    weights_only=True,
)

vae = VAE(latent_dim=vae_checkpoint["latent_dim"])
vae.load_state_dict(vae_checkpoint["model_state_dict"])
vae = vae.to(device).eval()
vae.requires_grad_(False)

memory = MDNRNN(**memory_checkpoint["model_config"])
memory.load_state_dict(memory_checkpoint["model_state_dict"])
memory = memory.to(device).eval()
memory.requires_grad_(False)

encoder = partial(encode_observation, vae=vae)

print("VAE epoch:", vae_checkpoint["epoch"])
print("Memory epoch:", memory_checkpoint["epoch"])

VAE epoch: 5
Memory epoch: 10


In [3]:
from torch.nn.utils import parameters_to_vector, vector_to_parameters

torch.manual_seed(0)

controller = Controller(
    latent_dim=memory_checkpoint["model_config"]["latent_dim"],
    hidden_dim=memory_checkpoint["model_config"]["hidden_dim"],
).to(device).eval()

# We'll optimize it externally rather than with backpropagation.
controller.requires_grad_(False)


def get_controller_vector(controller):
    return (
        parameters_to_vector(controller.parameters())
        .detach()
        .cpu()
        .numpy()
        .astype(np.float64, copy=True)
    )


@torch.no_grad()
def set_controller_vector(controller, values):
    reference = next(controller.parameters())
    vector = torch.as_tensor(
        np.asarray(values),
        dtype=reference.dtype,
        device=reference.device,
    ).clone()

    expected = sum(p.numel() for p in controller.parameters())
    if vector.ndim != 1 or vector.numel() != expected:
        raise ValueError(f"Expected a vector of length {expected}.")
    if not torch.isfinite(vector).all():
        raise ValueError("Controller parameters must be finite.")

    vector_to_parameters(vector, controller.parameters())


initial_vector = get_controller_vector(controller)
print("Parameters to optimize:", initial_vector.size)  # 867

Parameters to optimize: 867


In [4]:
def evaluate_controller(controller, seeds, max_steps=1000):
    returns = []
    lengths = []

    env = gym.make(
        "CarRacing-v3",
        continuous=True,
        domain_randomize=False,
    )

    try:
        for seed in seeds:
            # CPU-only inference here; preserve the caller's RNG state.
            with torch.random.fork_rng(devices=[]):
                torch.manual_seed(100_000 + int(seed))

                result = rollout(
                    env=env,
                    encode_observation=encoder,
                    memory=memory,
                    controller=controller,
                    seed=int(seed),
                    max_steps=max_steps,
                )

            returns.append(float(result["reward"]))
            lengths.append(int(result["steps"]))
    finally:
        env.close()

    return {
        "mean_reward": float(np.mean(returns)),
        "returns": returns,
        "steps": lengths,
    }

In [5]:
development_seeds = [100, 101, 102]

baseline = evaluate_controller(
    controller,
    seeds=development_seeds,
)

print("Per-track rewards:", baseline["returns"])
print("Episode lengths:", baseline["steps"])
print("Mean reward:", baseline["mean_reward"])

run_id = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
controller_run_dir = repo_root / "runs" / f"controller_pilot_{run_id}"
controller_run_dir.mkdir(parents=True, exist_ok=False)

torch.save(
    {
        "model_state_dict": controller.state_dict(),
        "memory_checkpoint": str(memory_path),
        "vae_checkpoint": str(vae_path),
        "development_seeds": development_seeds,
        "baseline": baseline,
    },
    controller_run_dir / "initial.pt",
)

(controller_run_dir / "baseline.json").write_text(
    json.dumps(baseline, indent=2),
    encoding="utf-8",
)

Per-track rewards: [-77.77777777777762, -83.49834983498299, -82.07885304659455]
Episode lengths: [1000, 1000, 1000]
Mean reward: -81.11832688645171


177

In [7]:
import cma

search_config = {
    "population_size": 8,
    "generations": 5,
    "sigma0": 0.1,
    "search_seed": 42,
    "development_seeds": list(development_seeds),
    "max_steps": 1000,
}

# Restore the exact controller used for the saved baseline.
initial_checkpoint = torch.load(
    controller_run_dir / "initial.pt",
    map_location="cpu",
    weights_only=True,
)
controller.load_state_dict(initial_checkpoint["model_state_dict"])

initial_vector = get_controller_vector(controller)
baseline = initial_checkpoint["baseline"]

assert list(development_seeds) == initial_checkpoint["development_seeds"]

# Each execution gets its own search directory.
search_id = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
search_dir = controller_run_dir / f"cma_{search_id}"
search_dir.mkdir(parents=True, exist_ok=False)

es = cma.CMAEvolutionStrategy(
    initial_vector.tolist(),
    search_config["sigma0"],
    {
        "popsize": search_config["population_size"],
        "seed": search_config["search_seed"],
        "maxiter": search_config["generations"],
        "verbose": -9,
        "verb_log": 0,
    },
)

# Include the baseline so we never replace it with a worse candidate.
best_vector = initial_vector.copy()
best_score = float(baseline["mean_reward"])
best_result = baseline
search_history = []

(search_dir / "config.json").write_text(
    json.dumps(
        {**search_config, "cma_version": str(cma.__version__)},
        indent=2,
    ),
    encoding="utf-8",
)

print("Starting development reward:", best_score)
print("Search directory:", search_dir)

Starting development reward: -81.11832688645171
Search directory: /Users/cedric/Repos/world-models-reproduction/runs/controller_pilot_20260910_194144_318051/cma_20260910_194236_077376


In [8]:
def save_best_controller():
    torch.save(
        {
            "parameter_vector": torch.tensor(
                best_vector, dtype=torch.float32
            ),
            "development_result": best_result,
            "memory_checkpoint": str(memory_path),
            "vae_checkpoint": str(vae_path),
            "search_config": search_config,
        },
        search_dir / "best.pt",
    )


save_best_controller()

In [9]:
for generation in range(1, search_config["generations"] + 1):
    if es.stop():
        print("CMA-ES stopping condition:", es.stop())
        break

    candidates = es.ask()
    fitnesses = []
    generation_rewards = []

    for candidate_index, vector in enumerate(candidates, start=1):
        set_controller_vector(controller, vector)

        result = evaluate_controller(
            controller,
            seeds=development_seeds,
            max_steps=search_config["max_steps"],
        )

        reward = float(result["mean_reward"])
        if not np.isfinite(reward):
            raise RuntimeError("Non-finite candidate reward.")

        fitnesses.append(-reward)
        generation_rewards.append(reward)

        if reward > best_score:
            best_score = reward
            best_vector = np.asarray(vector).copy()
            best_result = result
            save_best_controller()

        search_history.append({
            "generation": generation,
            "candidate": candidate_index,
            "mean_reward": reward,
            "returns": result["returns"],
            "steps": result["steps"],
        })

        (search_dir / "history.json").write_text(
            json.dumps(search_history, indent=2),
            encoding="utf-8",
        )

        print(
            f"Generation {generation:02d} | "
            f"candidate {candidate_index:02d}/{len(candidates)} | "
            f"reward={reward:.2f} | best={best_score:.2f}",
            flush=True,
        )

    # Update the search distribution after evaluating the population.
    es.tell(candidates, fitnesses)

    print(
        f"Generation {generation:02d} complete | "
        f"population mean={np.mean(generation_rewards):.2f} | "
        f"population best={np.max(generation_rewards):.2f}",
        flush=True,
    )

set_controller_vector(controller, best_vector)

print("\nInitial development reward:", baseline["mean_reward"])
print("Best development reward:", best_score)
print("Best per-track rewards:", best_result["returns"])

Generation 01 | candidate 01/8 | reward=-83.59 | best=-81.12
Generation 01 | candidate 02/8 | reward=-75.29 | best=-75.29
Generation 01 | candidate 03/8 | reward=-70.37 | best=-70.37
Generation 01 | candidate 04/8 | reward=-78.77 | best=-70.37
Generation 01 | candidate 05/8 | reward=-45.49 | best=-45.49
Generation 01 | candidate 06/8 | reward=5.61 | best=5.61
Generation 01 | candidate 07/8 | reward=-31.59 | best=5.61
Generation 01 | candidate 08/8 | reward=-69.16 | best=5.61
Generation 01 complete | population mean=-56.08 | population best=5.61
Generation 02 | candidate 01/8 | reward=-28.04 | best=5.61
Generation 02 | candidate 02/8 | reward=-64.80 | best=5.61
Generation 02 | candidate 03/8 | reward=-42.39 | best=5.61
Generation 02 | candidate 04/8 | reward=-49.17 | best=5.61
Generation 02 | candidate 05/8 | reward=-18.74 | best=5.61
Generation 02 | candidate 06/8 | reward=-74.29 | best=5.61
Generation 02 | candidate 07/8 | reward=-38.99 | best=5.61
Generation 02 | candidate 08/8 | rew

In [10]:
heldout_seeds = [200, 201, 202, 203, 204]
assert set(heldout_seeds).isdisjoint(development_seeds)

try:
    set_controller_vector(controller, initial_vector)
    initial_heldout = evaluate_controller(
        controller, seeds=heldout_seeds
    )

    set_controller_vector(controller, best_vector)
    best_heldout = evaluate_controller(
        controller, seeds=heldout_seeds
    )
finally:
    set_controller_vector(controller, best_vector)

comparison = {
    "seeds": heldout_seeds,
    "initial": initial_heldout,
    "selected": best_heldout,
}

(search_dir / "heldout_comparison.json").write_text(
    json.dumps(comparison, indent=2),
    encoding="utf-8",
)

for seed, before, after in zip(
    heldout_seeds,
    initial_heldout["returns"],
    best_heldout["returns"],
):
    print(
        f"Track {seed}: "
        f"initial={before:.2f}, selected={after:.2f}, "
        f"change={after - before:+.2f}"
    )

print("\nInitial held-out mean:", initial_heldout["mean_reward"])
print("Selected held-out mean:", best_heldout["mean_reward"])

Track 200: initial=-81.07, selected=13.56, change=+94.64
Track 201: initial=-77.70, selected=-7.06, change=+70.63
Track 202: initial=-78.26, selected=23.19, change=+101.45
Track 203: initial=-83.33, selected=25.00, change=+108.33
Track 204: initial=-81.13, selected=39.62, change=+120.75

Initial held-out mean: -80.29880017230798
Selected held-out mean: 18.862503810045375
